In [77]:
import pandas as pd
import numpy as np
import mne
import pywt
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

In [78]:
file_path = "Original_Data.xlsx"
ecbl_data = pd.read_excel(file_path, sheet_name="ECBL")
eobl_data = pd.read_excel(file_path, sheet_name="EOBL")

In [79]:
# Sampling frequency
sfreq = 512  # Hz

In [80]:
# filter for alpha spindles (8-13 Hz)
def bandpass_filter(data, sfreq, low=8, high=13):
    info = mne.create_info(ch_names=list(data.columns[1:]), sfreq=sfreq, ch_types="eeg")
    raw = mne.io.RawArray(data.iloc[:, 1:].values.T, info)
    raw.filter(low, high, fir_design='firwin')
    return raw.get_data()

In [81]:
# Apply filter
ecbl_filtered = bandpass_filter(ecbl_data, sfreq)
eobl_filtered = bandpass_filter(eobl_data, sfreq)

Creating RawArray with float64 data, n_channels=30, n_times=76001
    Range : 0 ... 76000 =      0.000 ...   148.438 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 13 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 13.00 Hz
- Upper transition bandwidth: 3.25 Hz (-6 dB cutoff frequency: 14.62 Hz)
- Filter length: 845 samples (1.650 s)

Creating RawArray with float64 data, n_channels=30, n_times=76001
    Range : 0 ... 76000 =      0.000 ...   148.438 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 13 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpa

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


In [82]:
# Function to count spindles using peak detection
def count_spindles(filtered_data):
    spindle_counts = {}
    for i, electrode in enumerate(filtered_data):
        peaks, _ = find_peaks(electrode, height=0.5, distance=sfreq//10) 
        spindle_counts[f"Electrode {i+1}"] = len(peaks)
    return spindle_counts

In [83]:
ecbl_spindle_counts = count_spindles(ecbl_filtered)
eobl_spindle_counts = count_spindles(eobl_filtered)

In [84]:
# Compare spindle counts
comparison_df = pd.DataFrame({"ECBL": ecbl_spindle_counts.values(), "EOBL": eobl_spindle_counts.values()},
                             index=ecbl_spindle_counts.keys())
print(comparison_df)

              ECBL  EOBL
Electrode 1    839   845
Electrode 2    826   847
Electrode 3    813   868
Electrode 4    854   883
Electrode 5    823   863
Electrode 6    862   901
Electrode 7    866   901
Electrode 8    898   911
Electrode 9    888   924
Electrode 10   849   874
Electrode 11   842   838
Electrode 12   868   860
Electrode 13   882   870
Electrode 14   876   913
Electrode 15   906   931
Electrode 16   915   920
Electrode 17   876   909
Electrode 18   887   927
Electrode 19   806   908
Electrode 20   775   895
Electrode 21   824   904
Electrode 22   819   884
Electrode 23   791   892
Electrode 24   848   917
Electrode 25   792   895
Electrode 26   813   879
Electrode 27   891   896
Electrode 28   874   872
Electrode 29   852   907
Electrode 30   874   918


In [ ]:
# Wavelet Transform and Scaleogram Visualization
def wavelet_transform_all_electrodes(data, sfreq):
    num_electrodes = data.shape[0]
    scales = np.arange(1, 128)  # Define scale range
    plt.figure(figsize=(12, 6))

    for i in range(num_electrodes):
        coef, freqs = pywt.cwt(data[i], scales, 'morl')  
        plt.imshow(coef, aspect='auto', cmap='coolwarm',
                   extent=[0, len(data[i]) / sfreq, min(scales), max(scales)],
                   alpha=0.7)  # Overlay with transparency

    plt.colorbar(label="Amplitude")
    plt.xlabel("Time (s)")
    plt.ylabel("Scale")
    plt.title("Wavelet Transform Scaleogram - All Electrodes")
    plt.show()

In [ ]:
# Call function for visualization
wavelet_transform_all_electrodes(ecbl_filtered, sfreq)

In [ ]:
# Save results
comparison_df.to_csv("alpha_spindle_comparison.csv", index=True)